# 1. Preprocess: 표준화 CSV → analytic_sample.csv

| 항목 | 내용 |
|------|------|
| **입력** | `output/pre_output/kwcs_pooled_raw.csv` (0_extract 출력) |
| **출력** | `output/pre_output/analytic_sample.csv` |
| **처리** | Complete-case 제외 + 파생변수 생성 + STROBE flow |

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 1 — 환경 설정 + 데이터 로드
# ══════════════════════════════════════════════════════════════
# Environment setup (adjust paths as needed)

import pandas as pd, numpy as np, os, hashlib

BASE    = os.path.abspath(os.path.join('..'))  # repository root
PRE_DIR = os.path.join(BASE, 'output', 'pre_output')

df = pd.read_csv(os.path.join(PRE_DIR, 'kwcs_pooled_raw.csv'), encoding='utf-8-sig')
print(f'로드 완료: N={len(df):,} (W2-7 풀링)')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 2 — Complete-case + 파생변수 + 저장
# ══════════════════════════════════════════════════════════════

N_pooled = len(df)

# ──────────────────────────────────────────────────────────────
# STEP 1: Complete-case 분석 (STROBE Flow)
# ──────────────────────────────────────────────────────────────
# [D4] Complete-case 채택 근거:
#   1. 반복횡단면(패널 아님) → 시간 연계 결측 패턴 추정 불가
#   2. 주요 결측 = 구조적 결측(7=해당없음) → imputation 부적절
#   3. Item-level 무응답(8/9) < 5%
#
# [D5] 2단계 제외 순서:
#   1단계: 결과+노출+인구통계, 2단계: 인체공학적 공변량
#   - auto/support는 skipna로 산출 가능 → key_vars에 미포함 (표본 보존)
#
# [D5a] erg_lift_ppl 제외: 결측률 62.3%, 간호/돌봄 특수 노출
#   → 4개 erg로 교란 보정, sensitivity에서 5개 포함 분석 별도 제시
# ──────────────────────────────────────────────────────────────

key_vars = ['msd_back','msd_upper','msd_lower',  # 결과변수
            'psy_demand',                         # 주요 노출
            'gender','age_raw','emp_type','whrs','wt']  # 인구통계/가중치

erg_vars = ['erg_posture','erg_heavy','erg_stand','erg_repeat']  # D5a: erg_lift_ppl 제외

n0 = len(df)
df = df.dropna(subset=key_vars)
n1 = len(df)
print(f'STEP 1a  핵심변수 결측 제거: {n0:,} → {n1:,} (제외 {n0-n1:,})')

df = df.dropna(subset=erg_vars)
n2 = len(df)
print(f'STEP 1b  인체공학 결측 제거: {n1:,} → {n2:,} (제외 {n1-n2:,})')


# ──────────────────────────────────────────────────────────────
# STEP 2: 파생 변수 (Derived Variables)
# ──────────────────────────────────────────────────────────────

# ── 2a. 결과변수 이진화 ──
# heal_prob: 1=Yes(지난 12개월 통증), 2=No → 1/0
df['msd_back_b']  = (df['msd_back']==1).astype(float)
df['msd_upper_b'] = (df['msd_upper']==1).astype(float)
df['msd_lower_b'] = (df['msd_lower']==1).astype(float)
df['msd_any']     = df[['msd_back_b','msd_upper_b','msd_lower_b']].max(axis=1)


# ── 2b. Karasek JDC-S 모델 ──
# ┌──────────────────────────────────────────────────────────────┐
# │ [D7] 척도 방향 정리 (KWCS 코드북 확인 완료)                   │
# │                                                              │
# │ hazard_psy1 (psy_demand): "시간적 압박 하에 일하는 빈도"       │
# │   1=항상(높은요구) ~ 6=거의없음(낮은요구)                      │
# │   → 낮은 값 = 높은 심리적 요구 (역방향 척도!)                  │
# │   → 고요구 = <= 25th percentile                              │
# │                                                              │
# │ decla1-3 (auto): "업무 순서/방법/속도 결정 가능 여부"           │
# │   1=Yes(자율있음), 2=No(없음)                                 │
# │   → 높은 mean = 낮은 자율성                                   │
# │   → 저자율 = mean >= 75th pctl                               │
# │                                                              │
# │ wsituation1-2 (sup_col/mgr): "동료/상사의 도움/지지 정도"      │
# │   1=항상(높은지지) ~ 5=전혀없음(낮은지지)                      │
# │   → 높은 값 = 낮은 지지                                       │
# │   → 저지지 = mean >= 75th pctl                               │
# │                                                              │
# │ hazard_psy2 (int_speed): "빠른 속도로 일하는 빈도"             │
# │   1=항상(높은강도) ~ 6=거의없음(낮은강도)                      │
# │   → 낮은 값 = 높은 작업 강도 (역방향 척도!)                    │
# │   → 고강도 = <= 25th pctl                                    │
# │                                                              │
# │ hazard_erg (erg_*): "해당 자세/동작 빈도"                      │
# │   1=항상(높은노출) ~ 6=거의없음(낮은노출)                      │
# │   → 낮은 값 = 높은 노출 (역방향 척도!)                        │
# │   → 고노출 = <= 25th pctl                                    │
# └──────────────────────────────────────────────────────────────┘

# 심리적 요구: 낮은값 = 높은요구 → 25th pctl 이하 = 고요구
psy_p25 = df['psy_demand'].quantile(0.25)
print(f'\npsy_demand 25th pctl = {psy_p25} (<=이 값 = HIGH demand)')
df['psy_demand_hi'] = (df['psy_demand'] <= psy_p25).astype(float)

# 의사결정 자율성: 1=Yes, 2=No → mean 높을수록 자율 낮음
df['autonomy'] = df[['auto1','auto2','auto3']].mean(axis=1, skipna=True)
auto_p75 = df['autonomy'].dropna().quantile(0.75)
print(f'autonomy 75th pctl = {auto_p75} (>=이 값 = LOW autonomy)')
df['lo_auto'] = np.where(df['autonomy'].notna(),
                          (df['autonomy'] >= auto_p75).astype(float), np.nan)

# 사회적 지지: 1=항상 ~ 5=전혀없음 → 높을수록 지지 낮음
df['support'] = df[['sup_col','sup_mgr']].mean(axis=1, skipna=True)
sup_p75 = df['support'].dropna().quantile(0.75)
print(f'support 75th pctl = {sup_p75} (>=이 값 = LOW support)')
df['lo_sup'] = np.where(df['support'].notna(),
                         (df['support'] >= sup_p75).astype(float), np.nan)

# 작업 강도: 낮은값 = 높은강도 → 25th pctl 이하 = 고강도
int_p25 = df['int_speed'].dropna().quantile(0.25)
print(f'int_speed 25th pctl = {int_p25} (<=이 값 = HIGH intensity)')
df['int_speed_hi'] = np.where(df['int_speed'].notna(),
                               (df['int_speed'] <= int_p25).astype(float), np.nan)


# ── 2c. 복합 지표 ──
# [D8] Job strain = High demand AND Low autonomy (Karasek, 1979)
#       Iso-strain = Job strain AND Low support (Johnson & Hall, 1988)
df['job_strain'] = ((df['psy_demand_hi']==1) & (df['lo_auto']==1)).astype(float)
df['iso_strain'] = ((df['job_strain']==1) & (df['lo_sup']==1)).astype(float)

# 일-생활 균형: 1=매우좋음 ~ 4=매우나쁨, >=3 = 불균형
df['poor_wlb'] = np.where(df['wlb'].notna(), (df['wlb']>=3).astype(float), np.nan)


# ── 2d. 인체공학적 노출 ──
# 1=항상(고노출) ~ 6=거의없음(저노출) → <= 25th pctl = 고노출
erg_cuts = {}
for v in erg_vars:
    p25 = df[v].quantile(0.25)
    erg_cuts[v] = p25
    df[f'{v}_hi'] = (df[v] <= p25).astype(float)
# erg_lift_ppl (sensitivity용)
if 'erg_lift_ppl' in df.columns:
    p25_lift = df['erg_lift_ppl'].dropna().quantile(0.25)
    df['erg_lift_ppl_hi'] = np.where(df['erg_lift_ppl'].notna(),
                                      (df['erg_lift_ppl'] <= p25_lift).astype(float), np.nan)
    erg_cuts['erg_lift_ppl (sens.)'] = p25_lift
print(f'Ergonomic 25th pctl cutoffs: {erg_cuts}')


# ── 2e. 공변량 ──
df['female'] = (df['gender']==2).astype(float)

age_m = df['age_raw'].mean()
age_s = df['age_raw'].std()
df['age_c']  = (df['age_raw'] - age_m) / age_s
df['age_c2'] = df['age_c'] ** 2
print(f'Age centering: mean={age_m:.4f}, sd={age_s:.4f}')

edu_p75 = df['edu'].dropna().quantile(0.75)
print(f'edu 75th pctl = {edu_p75}')
df['edu_hi'] = np.where(df['edu'].notna(), (df['edu']>=edu_p75).astype(float), np.nan)

df['long_hrs'] = (df['whrs'] > 52).astype(float)
df['wage'] = (df['emp_type']==1).astype(float)


# ── 2f. 웨이브 변수 ──
for w in range(2,8):
    df[f'wd{w}'] = (df['wave']==w).astype(float)
df['wave_c'] = (df['wave']-1) / 6
df['swt']    = df['wt']


# ──────────────────────────────────────────────────────────────
# STEP 3: 저장 + 검증
# ──────────────────────────────────────────────────────────────
FINAL_COLS = [
    'wave','year',
    'msd_back','msd_upper','msd_lower',
    'psy_demand','auto1','auto2','auto3','sup_col','sup_mgr','int_speed',
    'erg_posture','erg_lift_ppl','erg_heavy','erg_stand','erg_repeat',
    'wlb','gender','age_raw','edu','emp_type','industry','whrs','wt',
    'msd_back_b','msd_upper_b','msd_lower_b','msd_any',
    'psy_demand_hi','autonomy','lo_auto','support','lo_sup','int_speed_hi',
    'job_strain','iso_strain','poor_wlb',
    'erg_posture_hi','erg_lift_ppl_hi','erg_heavy_hi','erg_stand_hi','erg_repeat_hi',
    'female','age_c','age_c2','edu_hi','long_hrs','wage',
    'wd2','wd3','wd4','wd5','wd6','wd7','wave_c','swt']

df_out = df[FINAL_COLS].sort_values(['wave','age_raw','gender']).reset_index(drop=True)

out = os.path.join(PRE_DIR, 'analytic_sample.csv')
df_out.to_csv(out, index=False, encoding='utf-8-sig')
with open(out,'rb') as f: sha = hashlib.sha256(f.read()).hexdigest()[:16]

# ── STROBE Flow + 기술통계 ──
YEARS = {2:2010, 3:2011, 4:2014, 5:2017, 6:2020, 7:2023}
print('\n' + '='*60)
print('STROBE Flow Summary')
print('='*60)
print(f'  Total pooled (W2-7):   {N_pooled:>10,}')
print(f'  - Missing key vars:    {N_pooled-n1:>10,}')
print(f'  After key exclusion:   {n1:>10,}')
print(f'  - Missing ergonomic:   {n1-n2:>10,}')
print(f'    (4 vars; erg_lift_ppl → sensitivity only)')
print(f'  Final analytic sample: {n2:>10,}')
print(f'\n  SHA-256: {sha}')
print(f'  Output:  {out}')

print('\nWave-specific N:')
for w in sorted(df_out['wave'].unique()):
    print(f'  W{int(w)} ({YEARS[int(w)]}): {(df_out["wave"]==w).sum():>7,}')

# ── 척도 방향 검증 (Sanity Check) ──
print('\n' + '='*60)
print('Scale Direction Sanity Check')
print('='*60)
print('기대: 고위험 노출 → 높은 MSD 유병률')
for var, label in [('psy_demand_hi','Hi demand'), ('lo_auto','Lo autonomy'),
                    ('lo_sup','Lo support'), ('job_strain','Job strain')]:
    col = df_out[var].dropna()
    msd = df_out.loc[col.index, 'msd_any']
    r1 = msd[col==1].mean()*100
    r0 = msd[col==0].mean()*100
    direction = 'OK (risk)' if r1 > r0 else 'REVERSE!' if r1 < r0 else 'FLAT'
    print(f'  {label:15s}: exposed={r1:.1f}%  unexposed={r0:.1f}%  → {direction}')

print('\nDescriptive Statistics (Table 1):')
N = len(df_out)
for name, val, pct in [
    ('N', f'{N:,}', ''),
    ('Female', f'{df_out["female"].sum():,.0f}', f'{df_out["female"].mean()*100:.1f}'),
    ('Age (mean,SD)', f'{df_out["age_raw"].mean():.1f}', f'SD={df_out["age_raw"].std():.1f}'),
    ('Long hrs >52', f'{df_out["long_hrs"].sum():,.0f}', f'{df_out["long_hrs"].mean()*100:.1f}'),
    ('Wage worker', f'{df_out["wage"].sum():,.0f}', f'{df_out["wage"].mean()*100:.1f}'),
    ('--- MSD ---', '', ''),
    ('Any MSD', f'{df_out["msd_any"].sum():,.0f}', f'{df_out["msd_any"].mean()*100:.1f}'),
    ('Back pain', f'{df_out["msd_back_b"].sum():,.0f}', f'{df_out["msd_back_b"].mean()*100:.1f}'),
    ('Upper limb', f'{df_out["msd_upper_b"].sum():,.0f}', f'{df_out["msd_upper_b"].mean()*100:.1f}'),
    ('Lower limb', f'{df_out["msd_lower_b"].sum():,.0f}', f'{df_out["msd_lower_b"].mean()*100:.1f}'),
    ('--- Psychosocial ---', '', ''),
    ('Hi demand', f'{df_out["psy_demand_hi"].sum():,.0f}', f'{df_out["psy_demand_hi"].mean()*100:.1f}'),
    ('Lo autonomy', f'{df_out["lo_auto"].dropna().sum():,.0f}', f'{df_out["lo_auto"].dropna().mean()*100:.1f}'),
    ('Lo support', f'{df_out["lo_sup"].dropna().sum():,.0f}', f'{df_out["lo_sup"].dropna().mean()*100:.1f}'),
    ('Job strain', f'{df_out["job_strain"].sum():,.0f}', f'{df_out["job_strain"].mean()*100:.1f}'),
    ('Iso-strain', f'{df_out["iso_strain"].sum():,.0f}', f'{df_out["iso_strain"].mean()*100:.1f}'),
]:
    pct_str = f'  ({pct}%)' if pct and '%' not in pct else f'  {pct}' if pct else ''
    print(f'  {name:<18} {val:>8}{pct_str}')
print('='*60)